In [1]:
!pip install torch_scatter torcheeg torch_geometric -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.4/251.4 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 16.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.1/295.1 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 57.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.utils as utils
from torch.utils.data import DataLoader, Subset ,WeightedRandomSampler
from torcheeg.models import CCNN
from torcheeg import transforms
from torcheeg.transforms import ToGrid
from torcheeg.datasets import SEEDIVDataset,SEEDIVFeatureDataset
from torcheeg.datasets.constants import SEED_IV_CHANNEL_LOCATION_DICT
from torcheeg.transforms import ToG
from torcheeg.datasets.constants import SEED_IV_ADJACENCY_MATRIX
from torcheeg.models import DGCNN
import torch_geometric.loader as geom_loader 
import copy
import scipy.signal as signal
import random
import numpy as np
import os

In [3]:
# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Set a fixed random seed for reproducibility across different libraries.
def set_seed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [5]:
def map_emotions(y):
    if y == 1 or y == 2:  # Sad or Fear
        return 0  # Negative
    if y == 3:  # Happy
        return 1  # Positive
    return -1 # Neutral

In [6]:
dataset = SEEDIVFeatureDataset(
    io_path='./tmp_out/seed_iv_features',
    root_path='/kaggle/input/seed-iv/eeg_feature_smooth',
    feature=['de_LDS'], 
    num_worker=0,
    offline_transform=transforms.Compose([
        transforms.To2d(), 
        transforms.Lambda(lambda x: torch.tensor(x).float())]),
    label_transform=transforms.Compose([
        transforms.Select('emotion'),
        transforms.Lambda(map_emotions)
    ])
   
)

[2026-03-02 22:49:39] INFO (torcheeg/MainThread) 🔍 | Processing EEG data. Processed EEG data has been cached to ./tmp_out/seed_iv_features.
[2026-03-02 22:49:39] INFO (torcheeg/MainThread) ⏳ | Monitoring the detailed processing of a record for debugging. The processing of other records will only be reported in percentage to keep it clean.
[PROCESS]:   0%|          | 0/45 [00:00<?, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 0it [00:00, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 1it [00:00,  7.62it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 22it [00:00, 112.71it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 46it [00:00, 166.64it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 70it [00:00, 193.73it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 93it [00:00, 204.83it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_201511

In [7]:
# This DataFrame contains columns like: subject_id, trial_id, emotion, etc.
meta_info = dataset.info 
all_subjects = sorted(meta_info['subject_id'].unique())
print(f"Subjects Found: {all_subjects}")

Subjects Found: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


In [8]:
# Parameters
EPOCHS = 100
BATCH_SIZE = 64
LR = 0.005
# PATIENCE_LR = 3
# REDUCE_FACTOR = 0.5
PATIENCE_ES = 25
WEIGHT_DECAY = 0.0001
HIDE_CHANNELS = 128
LABEL_SMOOTHING = 0.1
NUM_LAYERS = 3

In [9]:
# Subject-Independent Loop
from sklearn.model_selection import train_test_split

# Create a directory to save the best models
if not os.path.exists('saved_models_independent'):
    os.makedirs('saved_models_independent')

independent_results = {}

print("Start Subject-Independent Evaluation (Leave-One-Subject-Out)...")

# LOSO Loop
for test_subject_id in all_subjects:
    print(f"\n{'='*40}")
    print(f" TESTING ON SUBJECT: {test_subject_id}")
    
    
     # Define Training Subjects
    train_subjects = [s for s in all_subjects if s != test_subject_id]

    all_filtered_info = meta_info.copy()
    all_filtered_info['mapped'] = all_filtered_info['emotion'].apply(map_emotions)
    
    # Collect Training and Testing indices
    train_indices = all_filtered_info[
        (all_filtered_info['subject_id'].isin(train_subjects)) & 
        (all_filtered_info['mapped'] != -1)
    ].index.tolist()
    
    test_indices = all_filtered_info[
        (all_filtered_info['subject_id'] == test_subject_id) & 
        (all_filtered_info['mapped'] != -1)
    ].index.tolist()


    print(f"Training on {len(train_subjects)} subjects | Testing on subject {test_subject_id}")
    print(f"Train samples (Filtered): {len(train_indices)} | Test samples (Filtered): {len(test_indices)}")
    print(f"{'='*40}")
    
    # Build Dataset Subsets
    train_set = Subset(dataset, train_indices)
    test_set = Subset(dataset, test_indices)
    
    # Data Balancing (Weighted Sampler)
    y_train = meta_info.iloc[train_indices]['emotion'].values
    mapped_labels = [map_emotions(y) for y in y_train]
    class_counts = np.bincount( mapped_labels)
    class_weights = 1. / (class_counts + 1e-6)
    sample_weights = [class_weights[y] for y in  mapped_labels]
    
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    
    # Data Loaders
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=False, sampler=sampler, num_workers=0)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Initialize Model 
    model = DGCNN(
        in_channels=5, 
        num_electrodes=62, 
        num_layers=NUM_LAYERS, 
        hid_channels=HIDE_CHANNELS, 
        num_classes=2,
    ).to(device)
    
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    # Training Loop 
    best_val_acc = 0.0
    best_val_loss = 0.0
    counter = 0 
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for X, y in train_loader:
            X = X.to(device).float() 
            y = y.to(device).long()
            if len(X.shape) == 4: X = X.squeeze(1) 
          
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += y.size(0)
            train_correct += (predicted == y).sum().item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_train_acc = 100 * train_correct / train_total
               
        # Validation on the UNSEEN Subject
        model.eval()
        val_correct = 0 
        val_total = 0
        val_loss = 0.0
        with torch.no_grad():
            for X, y in test_loader:
                X = X.to(device).float()
                y = y.to(device).long()
                
                if len(X.shape) == 4: X = X.squeeze(1)
                outputs = model(X)

                loss = criterion(outputs, y)
                val_loss += loss.item()
                              
                _, predicted = torch.max(outputs.data, 1)
                val_total += y.size(0)
                val_correct += (predicted == y).sum().item()
        
        avg_val_loss = val_loss / len(test_loader)
        avg_val_acc = 100 * val_correct / val_total
        scheduler.step()

        # Print metrics for the current epoch to follow the progress
        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] -> "
              f"Train Loss: {avg_train_loss:.4f} ,Train Acc: {avg_train_acc:.2f}% | "
              f"Val Loss: {avg_val_loss:.4f} , Val Acc: {avg_val_acc:.2f}%")

        # Save the best model based on Validation Accuracy
        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            counter = 0
            torch.save(model.state_dict(), f"saved_models_independent/loso_test_subject_{test_subject_id}.pth")
        else:
            counter += 1
            if counter >= PATIENCE_ES:
                print(f"Early stopping triggered for Subject {test_subject_id} at epoch {epoch}")
                break
                
    print(f"Subject {test_subject_id} Best Acc: {best_val_acc:.2f}%")
    independent_results[test_subject_id] = best_val_acc


    del model, optimizer, train_loader, test_loader, train_set, test_set
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ================= FINAL REPORT =================
print("\n" + "="*60)
print("SUBJECT-INDEPENDENT (LOSO) FINAL SUMMARY")
print("="*60)

all_accuracies = list(independent_results.values())

for sub_id in sorted(independent_results.keys()):
    print(f"Subject {sub_id}: Accuracy = {independent_results[sub_id]:.2f}% ")

print(f"\nAverage Accuracy: {np.mean(all_accuracies):.2f}%")

if len(independent_results) > 0:
    best_sub_id = max(independent_results, key=independent_results.get)
    print(f"HIGHEST ACCURACY: Subject {best_sub_id} ({independent_results[best_sub_id]:.2f}%)")
print("="*60)

Start Subject-Independent Evaluation (Leave-One-Subject-Out)...

 TESTING ON SUBJECT: 1
Training on 14 subjects | Testing on subject 1
Train samples (Filtered): 25578 | Test samples (Filtered): 1827
Epoch [01/100] -> Train Loss: 0.5012 ,Train Acc: 80.57% | Val Loss: 0.4810 , Val Acc: 76.74%
Epoch [02/100] -> Train Loss: 0.3212 ,Train Acc: 93.62% | Val Loss: 0.5860 , Val Acc: 74.27%
Epoch [03/100] -> Train Loss: 0.2661 ,Train Acc: 97.28% | Val Loss: 0.6565 , Val Acc: 71.26%
Epoch [04/100] -> Train Loss: 0.2383 ,Train Acc: 98.98% | Val Loss: 0.6559 , Val Acc: 73.51%
Epoch [05/100] -> Train Loss: 0.2253 ,Train Acc: 99.48% | Val Loss: 0.6395 , Val Acc: 76.74%
Epoch [06/100] -> Train Loss: 0.2211 ,Train Acc: 99.61% | Val Loss: 0.5546 , Val Acc: 79.31%
Epoch [07/100] -> Train Loss: 0.2215 ,Train Acc: 99.55% | Val Loss: 0.6245 , Val Acc: 76.08%
Epoch [08/100] -> Train Loss: 0.2291 ,Train Acc: 99.00% | Val Loss: 0.5062 , Val Acc: 79.86%
Epoch [09/100] -> Train Loss: 0.2287 ,Train Acc: 99.11% |